# Demo | Capítulo X: [Tema]
**Equipo:** <nombre-equipo>

## Índice
1. Segun el contenido del capitulo --> teoría breve + ejemplo mínimo (sobre el dataset dado)
2. Análisis o interpretación de resultados en cada caso, que muestra eso para el escenario, selección de metodos y criterio de seleccción si corresponde
4. Gotchas / anti-patrones comunes
5. Cierre 

# SETUP
Antes de correr los notebooks, instalá las dependencias desde la terminal:

```bash
pip install -r requirements.txt
```

(Si estás en Kaggle Notebooks o Google Colab, no hace falta este paso.) (Si queres correrlo en el notebook agregar ! antes)

# SETUP --> Dataset

Segun la manera elegida de usar el dataset
https://www.kaggle.com/datasets/alejandroczernikier/properati-argentina-dataset

In [ ]:
import pandas as pd
import numpy as np
import os

def cargar_properati():
    # cargar el dataset Properati desde distintas fuentes, según el entorno donde se esté ejecutando el notebook.
   
    # 1) Si ya descargan el csv y queda localmente en data/ (repo clonado)
    '''
    ruta_local = "../data/entrenamiento.csv"
    if os.path.exists(ruta_local):
        print("Cargando desde data/ local")
        return pd.read_csv(ruta_local)'''

    # 2) Si estamos corriendo en Kaggle Notebooks --> con ir al dataset y abrir notebook tambien tienen pasos guiados
    '''
    ruta_kaggle = "/kaggle/input/datasets/alejandroczernikier/properati-argentina-dataset/entrenamiento.csv"
    if os.path.exists(ruta_kaggle):
        print("Cargando desde Kaggle Notebooks")
        return pd.read_csv(ruta_kaggle)'''

    # 3) Si no está en ningún lado, usar kagglehub para bajarlo
    #    (funciona en Kaggle, Colab y local, siempre que haya credenciales configuradas)
    # agregar a requirements kagglehub (!)
    try:
        import kagglehub
        path = kagglehub.dataset_download("alejandroczernikier/properati-argentina-dataset")
        print(f"Descargado con kagglehub en: {path}")
        return pd.read_csv(os.path.join(path, "entrenamiento.csv"))
    except Exception as e:
        raise FileNotFoundError(
            "No se encontró el dataset en ninguna fuente. "
            "Revisá el README para instrucciones de descarga según tu entorno. "
            f"Error original: {e}"
        )

df = cargar_properati()
print(df.shape)
df.head()

# El Modelo de Regresión Lineal
### Introducción
Presentamos la teoría de la Regresión Lineal Simple, adaptando los conceptos estadísticos al análisis del mercado inmobiliario argentino utilizando el conjunto de datos de Properati.
Nuestro objetivo es comprender y modelar la relación lineal existente entre:

**Variable Independiente (x):** La superficie de la propiedad (expresada en metros cuadrados, $m^2$).

**Variable Dependiente (y):** El precio normalizado de la propiedad (expresado en dólares, USD).

Para esto, comenzamos introduciendo el concepto principal:
El modelo de regresión lineal es una herramienta estadística diseñada para analizar la relación entre una variable independiente (x) y una variable dependiente (y). Esta técnica asume que existe una relación entre las variables en análisis, es decir, en este caso que el precio de una propiedad está determinado principalmente por su superficie, estableciendo una relación directa descrita por la siguiente ecuación matemática:

$\qquad y_i = \alpha + \beta x_i + \epsilon_i$

Donde:
*   $y_i$ (**Precio Real**): Es el precio observado en el anuncio de venta o alquiler $i$ del conjunto de datos de Properati.
*   $x_i$ (**Superficie**): Es la superficie medida en metros cuadrados de dicha propiedad $i$.
*   $\beta$ (**Pendiente o Beta**): Representa el costo incremental por metro cuadrado. Indica cuánto se espera que aumente el precio de la propiedad por cada m2 adicional de superficie.
*   $\alpha$ (**Intersección o Alpha**): Representa el "precio base" teórico de una propiedad cuando su superficie es cero. En la práctica, suele absorber costos fijos, tasas impositivas basales o costos del terreno que no dependen directamente de los metros cuadrados construidos.
*   $\epsilon_i$ (**Término de Error o Residuo**): Representa la desviación o discrepancia para la propiedad $i$. Captura todos los factores del mundo real que influyen en el precio y que nuestro modelo simple no puede capturar únicamente con los metros cuadrados (por ejemplo: la ubicación exacta, la antigüedad, la presencia de cochera, amenities, o la calidad de los materiales).

Una vez que hemos calculado los parámetros óptimos de $\alpha$ y $\beta$ para nuestro dataset, podemos realizar estimaciones de precios utilizando la fórmula de predicción:

$\qquad \hat{y}_i = \alpha + \beta x_i$

Donde $\hat{y}_i$ es el precio estimado o predicho por el modelo para una propiedad que mide exactamente $x_i$ metros cuadrados.

## 2. El Error de Predicción y la Suma de Errores al Cuadrado (RSS)
En la realidad, las propiedades no se tasan con perfección matemática absoluta; siempre existirá una diferencia entre lo que nuestro modelo predice y el precio real publicado en Properati. A esta diferencia la denominamos error de predicción (o residuo) de un anuncio individual:

$\qquad \text{Error}_i = y_i - \hat{y}_i = y_i - (\alpha + \beta x_i)$

Para evitar la cancelación de errores positivos y negativos, la estadística eleva cada diferencia al cuadrado antes de sumarlas. La fórmula general para el error acumulado en nuestro conjunto de datos Properati es:

$\qquad \text{Suma de Errores al Cuadrado} = \sum_{i=1}^{n} (y_i - (\alpha + \beta x_i))^2$

*   **Penalización de desvíos grandes:** Al elevar al cuadrado, los errores grandes se penalizan con mucha mayor severidad que los pequeños (por ejemplo, errar por 10 USD se convierte en una penalización de 100, pero errar por 100 USD se convierte en 10,000), forzando al modelo a evitar tasaciones extremadamente erróneas.
*   **Facilidad matemática:** Al eliminar los signos negativos, la función se vuelve suave y continua, lo que nos permite usar el cálculo y la optimización para encontrar sus mínimos absolutos de forma directa.

### 3. El Método de Mínimos Cuadrados
El Método de Mínimos Cuadrados es el procedimiento de optimización que busca, precisamente, encontrar los valores óptimos de $\alpha$ (intersección) y $\beta$ (pendiente) que logren reducir la Suma de Errores al Cuadrado al valor más bajo posible. Utilizando derivadas parciales, se obtienen las fórmulas exactas para calcular estos parámetros óptimos directamente desde nuestros datos de Properati:

La **Pendiente** ($\beta$): Determina cuánto cambia el precio de una propiedad en Properati por cada metro cuadrado adicional. Se calcula mediante la siguiente relación:

$\qquad \beta = \text{correlación}(x,y) \frac{\text{desviación estándar}(y)}{\text{desviación estándar}(x)}$

En esta fórmula, la correlación cuantifica la intensidad y dirección de la relación lineal entre superficie y precio: valores cercanos a 1 indican que al aumentar la superficie, el precio aumenta de forma predecible (relación positiva), mientras que valores cercanos a 0 indican una falta de relación clara.
Por su parte, la desviación estándar nos dice cuánto varían "típicamente" los datos respecto al promedio: la de $y$ mide la variación típica en dólares (USD) y la de $x$ la variación típica en metros cuadrados (m2). Al dividirlas ($\frac{\text{desviación}(y)}{\text{desviación}(x)}$), creamos un "tipo de cambio" que le enseña al modelo a hablar en dólares por metro cuadrado (USD/m2), permitiendo que la pendiente final tenga sentido financiero.

La **Intersección** ($\alpha$): Representa el punto de partida o el precio base estimado. Se calcula como:

$\qquad \alpha = \text{media}(y) - \beta \text{media}(x)$

Esta fórmula asegura que cuando una propiedad tiene una superficie exactamente igual al promedio general de Properati, el modelo predecirá para ella un precio equivalente al promedio del mercado.

### 4. El Coeficiente de Determinación (R2)
El R2 es la métrica que nos permite evaluar qué tan bien ajusta nuestro modelo a los datos comparando dos componentes:

*   **RSS (Suma de Errores al Cuadrado):** El error que nuestro modelo no logra explicar (lo que queda "suelto" o mal predicho).
*   **TSS (Suma Total de Cuadrados):** Es la suma total de cuadrados de las desviaciones de cada valor observado $y_i$ respecto a la media de $y$.

La fórmula es:

$\qquad R^2 = 1.0 - \frac{\text{Suma de Errores al Cuadrado (RSS)}}{\text{Suma Total de Cuadrados (TSS)}}$

El resultado mide qué porcentaje de la variación de precios en Properati explica la superficie:

*   **R2 = 0:** El modelo no aporta información útil; el error residual es igual a la variabilidad total.
*   **R2 = 1:** El modelo explica perfectamente toda la variación; no hay errores de predicción.

**Interpretación:** Un R2 de 0.65, por ejemplo, indica que el 65% de la variación en los precios de las propiedades se explica simplemente con la superficie, mientras que el 35% restante se debe a otros factores no incluidos en este modelo simple (como ubicación, amenities o antigüedad).

### 5. Optimización mediante Descenso del Gradiente
Cuando trabajamos con bases de datos masivas como la de Properati (cercana a 1 millón de anuncios), calcular las medias, desviaciones estándar y correlaciones de manera exacta en memoria puede ser computacionalmente pesado o requerir arquitecturas distribuidas. Una alternativa muy utilizada en Machine Learning es resolver los parámetros de manera iterativa utilizando Descenso del Gradiente.

Este algoritmo comienza con valores iniciales aleatorios para el "precio base" ($\alpha$) y el "precio por metro cuadrado" ($\beta$), y calcula paso a paso cómo se comporta el error total del modelo.

Para saber cómo ajustar estos parámetros y reducir el error en cada paso, calculamos las derivadas parciales de la función de pérdida (nuestra Suma de Errores al Cuadrado) con respecto a cada parámetro:

*   **Gradiente con respecto al precio base** ($\alpha$):

    $\qquad \frac{\partial \text{Loss}}{\partial \alpha} = \sum_{i=1}^{n} -2(\text{error}_i)$

*   **Gradiente con respecto al precio por metro cuadrado** ($\beta$):

    $\qquad \frac{\partial \text{Loss}}{\partial \beta} = \sum_{i=1}^{n} -2(\text{error}_i)x_i$

En cada iteración el algoritmo actualiza la estimación de los parámetros desplazándose en dirección opuesta al gradiente (hacia abajo en la curva de error), utilizando un factor de escala llamado **tasa de aprendizaje** ($\eta$ o learning rate):

$\qquad \alpha_{\text{nuevo}} = \alpha_{\text{viejo}} - \eta \frac{\partial \text{Loss}}{\partial \alpha}$

$\qquad \beta_{\text{nuevo}} = \beta_{\text{viejo}} - \eta \frac{\partial \text{Loss}}{\partial \beta}$

Este proceso se repite miles de veces hasta que los parámetros convergen y se estabilizan en valores muy cercanos a los de las fórmulas matemáticas explicadas anteriormente.

### 6. Estimación de Máxima Verosimilitud
La Estimación de Máxima Verosimilitud (MLE) es el sustento estadístico que da rigor a nuestro modelo. Dado que en la realidad no conocemos los verdaderos parámetros ocultos del mercado (el precio base $\alpha$ y el valor por m2 $\beta$), la meta de MLE es encontrar las mejores estimaciones de estos valores (calculadas por zona para reflejar cada realidad local) que hagan que los precios reales que vemos publicados en Properati sean lo más probables de haber ocurrido.

Para que esto funcione, asumimos que dentro de cada barrio los errores de tasación (la diferencia entre el precio real y la estimación) siguen una distribución normal en torno a cero: es decir, muchos desvíos pequeños y casi ninguno gigantesco. Si mezcláramos zonas de alto poder adquisitivo con zonas más económicas, este supuesto se rompería por las diferencias desproporcionadas de precios.

Al calcular matemáticamente esta probabilidad para toda la zona, la distribución normal nos demuestra que maximizar la verosimilitud del modelo equivale exactamente a minimizar la suma de sus errores al cuadrado (RSS). Así, el método de Mínimos Cuadrados deja de ser una elección arbitraria y se convierte en el único camino estadísticamente óptimo para encontrar la recta de precios de cada mercado local.

## 4. Gotchas / anti-patrones

- Gotcha 1
- Gotcha 2

## 5. Cierre

Pasamos al ejercicio guiado en `ejercicios_<equipo>.ipynb`.